In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import cvxpy as cp
import numpy as np
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

In [3]:
from lipschitz_decomon_tools import echantillonner_boule_l2_simple, square_backward_bounds
import sys
sys.path.append('..')
from data_processing import load_data

In [4]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST08")

In [5]:
x_sample = x_test[0:1].flatten()

In [6]:
label = (y_train[0]).argmax()
print(label)

1


In [7]:
model_path = "/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output.keras"
model = keras.models.load_model(model_path)
model.compile(
   
    loss=HKR(
        alpha=10.0, min_margin=1.0
    ),  # HKR stands for the hinge regularized KR loss
    metrics=[
        # KR,  # shows the KR term of the loss
        HingeMargin(min_margin=1.0),  # shows the hinge term of the loss
    ],
    optimizer=Adam(learning_rate=0.001),)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 12 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [8]:
# def square_backward_bounds(l, u, y):
#     # l (4,)
#     # u (4,)
#     # y (4,)

#     u = u - y
#     l = l - y

#     W = u + l #(4,)
#     # print(cp.multiply(-u,l).shape)
#     b =cp.sum(cp.multiply(-u,l)) - W@y #scalar
#     return W, np.array(b)[None]#(4,) & (1,)

In [9]:
def function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1):
    # function we want to optimize, combination of lipschitz constraints in all yi
    outputs = []
    for i in range(len(y_list)):
        if label == 0:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*cp.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            # concave
        else:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*cp.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)   
            # convexe
    if label == 0:
        # min(min(concaves)) -> min(concave) -> NON CONVEXE
        function = cp.min(cp.hstack(outputs)) 
    else:
        # min(max(convexes)) -> min(convexe) -> CONVEXE
        function = cp.max(cp.hstack(outputs))
    return function

In [39]:
eps = 0.1
nb_pts = 100

In [41]:
x = cp.Variable(784)

In [42]:
y_list = []
for _ in range(nb_pts):
        y_list.append(echantillonner_boule_l2_simple(x_sample, eps))

In [43]:
l = x_sample-eps
u = x_sample+eps
W_list = []
b_list = []
for y_i in y_list:
    W, b = square_backward_bounds(l,u,y_i)
    W_list.append(W)
    b_list.append(b)

In [44]:
constraints = [eps**2 - cp.norm(x - x_sample, 2)**2 >=0]
obj = cp.Minimize(function_to_optimize_all(x, label, W_list, b_list, y_list, model, L=1))

In [62]:
prob = cp.Problem(obj, constraints)
# prob.solve(solver='CLARABEL', verbose=True)  # Returns the optimal value.
# prob.solve(solver='ECOS', verbose=True)  # Returns the optimal value.
prob.solve(solver='SCS', verbose=True)  # Returns the optimal value.


(CVXPY) Jul 23 02:04:14 PM: Your problem has 784 variables, 1 constraints, and 0 parameters.
(CVXPY) Jul 23 02:04:14 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Jul 23 02:04:14 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Jul 23 02:04:14 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Jul 23 02:04:14 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Jul 23 02:04:14 PM: Compiling problem (target solver=SCS).
(CVXPY) Jul 23 02:04:14 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CVXPY) Jul 23 02:04:14 PM: Applying reduction Dcp2Cone


(CVXPY) Jul 23 02:04:14 PM: Applying reduction CvxAttr2Constr
(CVXPY) Jul 23 02:04:14 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.6.6                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Jul 23 02:04:15 PM: Applying reduction SCS
(CVXPY) Jul 23 02:04:15 PM: Finished problem compilation (took 4.190e-01 seconds).
(CVXPY) Jul 23 02:04:15 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.7 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 887, constraints m: 1189
cones: 	  l: linear vars: 101
	  q: soc vars: 1088, qsize: 102
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 157889, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s

(CVXPY) Jul 23 02:04:16 PM: Problem status: optimal
(CVXPY) Jul 23 02:04:16 PM: Optimal value: 1.086e+00
(CVXPY) Jul 23 02:04:16 PM: Compilation took 4.190e-01 seconds
(CVXPY) Jul 23 02:04:16 PM: Solver (including time spent in interface) took 1.192e+00 seconds


  1675| 2.11e-05  1.20e-05  1.69e-05  1.09e+00  3.20e-01  1.19e+00 
------------------------------------------------------------------
status:  solved
timings: total: 1.19e+00s = setup: 5.64e-02s + solve: 1.13e+00s
	 lin-sys: 1.05e+00s, cones: 8.50e-03s, accel: 2.91e-03s
------------------------------------------------------------------
objective = 1.085930
------------------------------------------------------------------
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


1.0859532627475668

In [47]:
x_tests = [echantillonner_boule_l2_simple(x_sample, eps) for _ in range(100)]

In [48]:
def function_to_optimize_all_np(x, label, W_list, b_list, y_list, model, L=1):
    # function we want to optimize, combination of lipschitz constraints in all yi
    outputs = []
    for i in range(len(y_list)):
        if label == 0:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] +\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)
            # concave
        else:
            output = model(y_list[i].reshape((1,28,28))[None]).cpu().detach().numpy()[0,0] -\
                L*np.sqrt(W_list[i]@x+b_list[i]) #scalar
            outputs.append(output)   
            # convexe
    if label == 0:
        # min(min(concaves)) -> min(concave) -> NON CONVEXE
        function = np.min(outputs)
    else:
        # min(max(convexes)) -> min(convexe) -> CONVEXE
        function = np.max(outputs)
    return function

In [49]:
list = [function_to_optimize_all_np(x, label, W_list, b_list, y_list, model, L=1) for x in x_tests]

In [50]:
min = np.min(list)

In [51]:
min

1.0885692980816528